# The Story

Final report notebook. This is the published artifact — interactive
Plotly charts, narrative prose, and a clear argument.

**Exported to HTML** in `reports/` for sharing.

---

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pipeline.config import load_config, get_data_processed_dir

PROJECT_NAME = "qol-immigration"
config = load_config(PROJECT_NAME)
processed_dir = get_data_processed_dir(config)

# Load panels
state = pd.read_parquet(processed_dir / "state_panel.parquet")
national = pd.read_parquet(processed_dir / "national_panel.parquet")
annual = state.groupby("year")[["median_household_income", "real_gdp_millions", "foreign_born_pct", "hpi_annual_avg"]].mean().reset_index()

## Headline

## Headline

**Median household income and GDP have diverged over time — and immigration-heavy states show a stronger gap.** This exploratory look at state and national data (1972–2025) surfaces correlations and shifts that deserve deeper, causal work.

## The Setup

## The Setup

We asked: *How do quality-of-life indicators (income, poverty, housing, health) move relative to immigration and economic growth across US states?* The pipeline combined Census/ACS, BEA, BLS, DHS, CDC, FHFA, and other sources into state-level and national panels, then ran a descriptive pass, flagging engine, and directed deep-dives (divergence, lag, threshold). What follows is a first story cut from those findings.

In [2]:
# Setup: national median income and real GDP over time (indexed to 100 at start)
base_row = annual[["median_household_income", "real_gdp_millions"]].dropna().iloc[0]
base_yr = annual.loc[annual["median_household_income"].first_valid_index(), "year"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=annual["year"], y=annual["median_household_income"] / base_row["median_household_income"] * 100, mode="lines+markers", name="Median household income (index)"))
fig.add_trace(go.Scatter(x=annual["year"], y=annual["real_gdp_millions"] / base_row["real_gdp_millions"] * 100, mode="lines+markers", name="Real GDP (index)"))
fig.update_layout(title=f"National trend: income vs GDP (base year = {int(base_yr)})", xaxis_title="Year", yaxis_title="Index", legend=dict(orientation="h"))
fig.show()

## Finding 1

## Finding 1: Income and GDP have diverged

When we index median household income and real GDP to 100 at the start of the series, income grows more slowly than GDP over time — a well-documented “decoupling.” The gap is visible in the national aggregate and is one of the core patterns the pipeline’s divergence deep-dive quantifies. Correlation at state level between foreign-born share and GDP is stronger than between foreign-born share and income, consistent with growth not translating one-for-one into household income in high-immigration states.

In [3]:
# Divergence: income vs GDP indexed; annotate year of max gap
base_row = annual[["median_household_income", "real_gdp_millions"]].dropna().iloc[0]
annual = annual.copy()
annual["income_idx"] = annual["median_household_income"] / base_row["median_household_income"] * 100
annual["gdp_idx"] = annual["real_gdp_millions"] / base_row["real_gdp_millions"] * 100
annual["gap"] = (annual["gdp_idx"] - annual["income_idx"]).abs()
valid = annual.dropna(subset=["gap"])
gap_year = int(valid.loc[valid["gap"].idxmax(), "year"]) if len(valid) else int(annual["year"].iloc[-1])
y_at_gap = float(valid.set_index("year").loc[gap_year, "income_idx"]) if gap_year in valid["year"].values else float(annual["income_idx"].iloc[-1])
fig = go.Figure()
fig.add_trace(go.Scatter(x=annual["year"], y=annual["income_idx"], mode="lines+markers", name="Median income (index)"))
fig.add_trace(go.Scatter(x=annual["year"], y=annual["gdp_idx"], mode="lines+markers", name="Real GDP (index)"))
fig.add_annotation(x=gap_year, y=y_at_gap, text=f"Max gap @ {gap_year}", showarrow=True)
fig.update_layout(title="Finding 1: Income vs GDP (indexed); divergence over time", xaxis_title="Year", yaxis_title="Index")
fig.show()

## Finding 2

## Finding 2: Immigration and voter registration move in opposite directions

At state level, higher foreign-born share is associated with *lower* voter registration (r ≈ −0.53). That doesn’t imply causation — it could reflect demographics, policy, or measurement — but it’s a striking pattern for the “dilution of voice” narrative and worth testing with controls and lag structure in future work.

In [4]:
# Finding 2: state-level foreign_born_pct vs voter_reg_total_pct (recent year)
recent = state[state["year"] >= 2015].groupby("state_name")[["foreign_born_pct", "voter_reg_total_pct"]].mean().reset_index()
fig = px.scatter(recent, x="foreign_born_pct", y="voter_reg_total_pct", text="state_name", trendline="ols")
fig.update_traces(textposition="top center")
fig.update_layout(title="Finding 2: Foreign-born % vs voter registration (state avg, 2015+)", xaxis_title="Foreign-born %", yaxis_title="Voter registration %")
fig.show()

## Finding 3

## Finding 3: Housing and immigration — correlation, not yet a clear threshold

State-level HPI and foreign-born share are positively correlated (r ≈ 0.49). The pipeline’s threshold deep-dive looks for a tipping point; the relationship in this sample is more linear than step-like. Causal mechanisms (demand, supply, policy) and metro-level data would sharpen the story.

In [5]:
# Finding 3: state-level foreign_born_pct vs hpi_annual_avg (recent year)
recent_h = state[state["year"] >= 2010].groupby("state_name")[["foreign_born_pct", "hpi_annual_avg"]].mean().reset_index()
fig = px.scatter(recent_h, x="foreign_born_pct", y="hpi_annual_avg", text="state_name", trendline="ols")
fig.update_traces(textposition="top center")
fig.update_layout(title="Finding 3: Foreign-born % vs HPI (state avg, 2010+)", xaxis_title="Foreign-born %", yaxis_title="HPI (annual avg)")
fig.show()

## The Takeaway

## The Takeaway

Income and GDP have diverged at the national level; state-level patterns are consistent with immigration correlating with both economic size and weaker income growth relative to that size, and with lower voter registration. This is exploratory, not causal. Next steps: add controls (education, industry mix, policy), run formal lag/panel models, and bring in metro-level housing and turnout data.

## Methodology & Limitations

## Methodology & Limitations

- **Data sources:** Census/ACS (income, poverty, education, foreign-born), BEA (state GDP), BLS (national earnings), FHFA (HPI), CDC (life expectancy), DHS (LPRs, refugees, naturalizations), Census (voter registration), NCES (graduation). See project `citations.md` and `config.yaml`.
- **Time range:** State panel 1972–2025 (indicators vary); national panel 1947–2025.
- **Geographic granularity:** 50 states + DC for state panel; national for national panel; district-level data in cross-section only.
- **Analytical approach:** Layer 1 descriptive (univariate, trends, changepoints, correlations, geo variation); Layer 2 flagging (correlation/changepoint/divergence/outlier flags); Layer 3 deep-dives (divergence, lag, threshold). Full notebooks: `01_descriptive`, `02_findings`, `03_deep_dive`.
- **Key limitations:** Correlation only; no causal claims. ACS 2020 disruptions; some state-year cells sparse. CIS noncitizen data is advocacy-source — cross-check with neutral sources.